# Chương 4.3 — Sensitivity Analysis của Cụm C: Quality + Edge Weight + Delta

11 tham số: 5 Q-weights (sum=1), ReasoningLenFull, AttachmentCountFull, WiMin, WiMax, Eta, Kappa.

Output từ `sensitivity bench --cluster=C`. Snapshot dùng ở đây (`snap_20260603_014800`) đã được populate quality signals từ `feedbackParsed`.

> **Tham số LIVE vs DORMANT trên dữ liệu chain-8453:**
>
> - **LIVE:** `Eta` (chi phối mạnh nhất — hệ số reward), `QWeightConfidence`, `QWeightReasoning`, `QWeightBreakdown`, `ReasoningLenFull`, `WiMax` (chỉ cận dưới khi WiMax<1.5). Hai trọng số `QWeightAttachment`/`QWeightPayment` tuy tín hiệu bằng 0 vẫn tác động *gián tiếp* qua mẫu số chuẩn hoá Q.
> - **DORMANT (≈0):** `AttachmentCountFull` (dữ liệu không có attachment), `WiMin` (sender-trust cố định = 100 → wᵢ tối thiểu = 0.6 > dải [0.1,0.5] nên không bao giờ chạm), `Kappa` (snapshot chỉ chứa service_feedback nên nhánh phạt không bao giờ chạy).
>
> Tín hiệu chất lượng thật trong dữ liệu: `reasoningLen` (từ `comment`, 1550/8229 feedback, dài 6–287 ký tự) và `hasRatingBreakdown` (từ `score`, 1527/8229).

In [ ]:
import sys
sys.path.insert(0, '..')
from lib import setup_thesis_style, save_figure, load_oat, load_tornado, load_sobol, load_grid

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

setup_thesis_style()

SNAP_ID = "snap_20260603_014800"  # snapshot ID from `sensitivity snapshot list`
OUTPUT_DIR = f"../output/{SNAP_ID}"
CLUSTER = "C"

In [ ]:
oat = load_oat(OUTPUT_DIR, CLUSTER)
params = oat["param"].unique()
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(11, 12), sharey=False)
axes = axes.flatten()
for ax, p in zip(axes, params):
    sub = oat[oat["param"] == p]
    ax.plot(sub["value"], sub["spearman"], marker="o", label="Spearman ρ")
    ax.axhline(0.95, ls="--", color="grey", lw=0.5)
    ax.set_title(p)
    ax.set_xlabel("value")
    ax.set_ylabel("ρ (rank stability)")
    ax.set_ylim(0, 1.05)
for ax in axes[len(params):]:
    ax.axis("off")
fig.suptitle("Fig 4.3.1 — OAT rank stability per parameter (Cluster C)")
fig.tight_layout()
save_figure(fig, "4_3_1_oat")
plt.show()

In [ ]:
tor = load_tornado(OUTPUT_DIR, CLUSTER)
tor["total"] = tor["delta_low"] + tor["delta_high"]
tor = tor.sort_values("total")
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.barh(tor["param"], -tor["delta_low"], color="#5b9bd5", label="Δ low (−)")
ax.barh(tor["param"], tor["delta_high"], color="#ed7d31", label="Δ high (+)")
ax.axvline(0, color="black", lw=0.5)
ax.set_title("Fig 4.3.2 — Tornado: |Δ mean trust| theo tham số (Cluster C)")
ax.set_xlabel("Δ điểm trust trung bình")
ax.legend(loc="lower right")
fig.tight_layout()
save_figure(fig, "4_3_2_tornado")
plt.show()

In [ ]:
qweights = ["QWeightReasoning", "QWeightAttachment", "QWeightBreakdown",
            "QWeightPayment", "QWeightConfidence"]
q_default = [0.20, 0.25, 0.20, 0.15, 0.20]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
fig, ax = plt.subplots(figsize=(7, 4))
bottom = 0.0
for w, val, col in zip(qweights, q_default, colors):
    ax.bar(["Default Q weights"], val, bottom=bottom, color=col, label=w)
    bottom += val
ax.set_ylabel("Weight")
ax.set_title("Fig 4.3.3 — Q-weight default composition (Σ = 1)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
fig.tight_layout()
save_figure(fig, "4_3_3_qweights_stacked")
plt.show()

In [ ]:
grid = load_grid(OUTPUT_DIR, CLUSTER)
param_cols = [c for c in grid.columns if c not in {"combo_id", "spearman", "kendall", "mean", "std"}]
print("Top-3 grid parameters:", param_cols)
# Choose the 2 axes with the widest mean-response spread; hold the 3rd at its median.
spreads = {p: grid.groupby(p)["mean"].mean().max() - grid.groupby(p)["mean"].mean().min()
           for p in param_cols}
y_ax, x_ax = sorted(param_cols, key=lambda p: spreads[p], reverse=True)[:2]
third = [p for p in param_cols if p not in (y_ax, x_ax)][0]
median3 = grid[third].median()
sub = grid[np.isclose(grid[third], median3, rtol=0.01)]
pivot = sub.pivot_table(index=y_ax, columns=x_ax, values="mean")
fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(pivot, cmap="viridis", ax=ax)
ax.set_title(f"Fig 4.3.4 — Mean trust over ({y_ax}, {x_ax}) at {third}={median3:g}")
fig.tight_layout()
save_figure(fig, "4_3_4_grid_heatmap")
plt.show()

In [ ]:
sob = load_sobol(OUTPUT_DIR, CLUSTER).sort_values("st", ascending=False)
print("Sobol total-order (ST) ranking — Cluster C:")
print(sob[["param", "s1", "st"]].to_string(index=False))